In [1]:
import pandas as pd
links =  pd.read_csv("data/patent.txt", sep = " ", names= ["source", "destination", "timestamp"])

In [2]:
links.head(5)

,source,destination,timestamp
0,0,1,0.000000
1,2,1,0.001241
2,3,1,0.041443
3,4,5,0.041559
4,6,1,0.041559


In [5]:
links.to_csv("data/patent.csv",index=None)

In [5]:
import pandas as pd

def edge_node_ratio(df: pd.DataFrame, directed: bool = False) -> float:
    """
    计算 edge-node ratio = |E| / |V|
    其中 |E| 是去重后的边数，不是 interaction 数量
    """
    # 节点数
    nodes = pd.unique(pd.concat([df["source"], df["destination"]], ignore_index=True))
    n = len(nodes)

    if n == 0:
        return 0.0

    edges = df[["source", "destination"]].copy()

    if directed:
        # 有向图：(u,v) 和 (v,u) 视为不同边
        m = len(edges.drop_duplicates())
    else:
        # 无向图：(u,v) 和 (v,u) 视为同一条边
        edges["u"] = edges[["source", "destination"]].min(axis=1)
        edges["v"] = edges[["source", "destination"]].max(axis=1)
        m = len(edges[["u", "v"]].drop_duplicates())

    return m / n

ratio = edge_node_ratio(links, directed=False)
print("edge-node ratio =", ratio)

edge-node ratio = 3.431799574259047


In [ ]:
import math
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import pandas as pd
import torch


@dataclass
class EdgeState:
    w: float
    last_t: float


class ExpDecayEdgeWeights:
    """
    Maintain undirected edge weights with lazy exponential decay:
      w_ij(t) = w_ij(last) * exp(-beta_e*(t-last)) + inc (when edge happens)
    """
    def __init__(self, beta_e: float):
        self.beta_e = float(beta_e)
        self.edges: Dict[Tuple[int, int], EdgeState] = {}

    def _key(self, u: int, v: int) -> Tuple[int, int]:
        return (u, v) if u <= v else (v, u)

    def touch(self, u: int, v: int, t: float, inc: float = 1.0) -> float:
        key = self._key(u, v)
        st = self.edges.get(key, None)
        if st is None:
            self.edges[key] = EdgeState(w=float(inc), last_t=float(t))
            return float(inc)

        dt = max(0.0, float(t) - st.last_t)
        if self.beta_e > 0 and dt > 0:
            st.w = st.w * math.exp(-self.beta_e * dt)
        st.w += float(inc)
        st.last_t = float(t)
        return st.w

    def decay_if_needed(self, key: Tuple[int, int], t: float) -> float:
        st = self.edges[key]
        dt = max(0.0, float(t) - st.last_t)
        if self.beta_e > 0 and dt > 0:
            st.w = st.w * math.exp(-self.beta_e * dt)
            st.last_t = float(t)
        return st.w

    def prune_small(self, eps: float = 1e-6):
        if eps <= 0:
            return
        to_del = [k for k, st in self.edges.items() if st.w < eps]
        for k in to_del:
            del self.edges[k]


# -------------------------
# 数值工具：正交化（QR）
# -------------------------
@torch.no_grad()
def orthonormalize_qr(W: torch.Tensor) -> torch.Tensor:
    # W: [n, k]
    # torch.linalg.qr returns Q with orthonormal columns
    Q, _ = torch.linalg.qr(W, mode="reduced")
    return Q


# -------------------------
# 核心：给定当前边权，计算 L_t W = D_t W - A_t W
# -------------------------
@torch.no_grad()
def laplacian_times_W(
    W: torch.Tensor,
    edge_weights: ExpDecayEdgeWeights,
    t: float,
    num_nodes: int,
    device: torch.device,
    dtype: torch.dtype,
    self_loop: bool = False,
    prune_eps: float = 1e-6,
) -> torch.Tensor:
    """
    Compute LW for unnormalized Laplacian L = D - A (undirected, weighted).
    We lazily decay edges to time t when iterating.
    Complexity ~ O(m_t * k).
    """
    n, k = W.shape
    assert n == num_nodes

    AW = torch.zeros((n, k), device=device, dtype=dtype)
    deg = torch.zeros((n,), device=device, dtype=dtype)

    # iterate edges
    for (u, v), st in list(edge_weights.edges.items()):
        w_uv = edge_weights.decay_if_needed((u, v), t)
        if w_uv < prune_eps:
            continue

        wu = float(w_uv)
        # A contributes: AW[u]+=w*W[v], AW[v]+=w*W[u]
        AW[u] += wu * W[v]
        AW[v] += wu * W[u]
        # degrees
        deg[u] += wu
        deg[v] += wu

    if self_loop:
        # optional: add identity to adjacency (rarely needed); keep off by default
        AW += W
        deg += 1.0

    DW = deg.view(-1, 1) * W
    LW = DW - AW

    # prune tiny edges occasionally
    edge_weights.prune_small(eps=prune_eps)
    return LW


# -------------------------
# 每个时间戳：Oja/扩散式更新 W_t
# -------------------------
@torch.no_grad()
def oja_update_Wt(
    W: torch.Tensor,
    edge_weights: ExpDecayEdgeWeights,
    t: float,
    num_nodes: int,
    alpha: float,
    iters: int,
    device: torch.device,
    dtype: torch.dtype,
    reorth_every: int = 1,
) -> torch.Tensor:
    """
    One timestamp update:
      repeat iters times:
        U = W - alpha * (L_t W)
        optionally re-orthonormalize columns
    """
    for s in range(iters):
        LW = laplacian_times_W(
            W=W,
            edge_weights=edge_weights,
            t=t,
            num_nodes=num_nodes,
            device=device,
            dtype=dtype,
        )
        W = W - float(alpha) * LW
        if reorth_every > 0 and ((s + 1) % reorth_every == 0):
            W = orthonormalize_qr(W)
    return W

def run_temporal_oja_from_csv(
    csv_path: str,
    k_dim: int = 16,
    alpha: float = 0.05,
    iters_per_batch: int = 2,
    beta_edge: float = 0.05,
    device: str = "cpu",
    dtype: torch.dtype = torch.float32,
    store_all_W: bool = True,
    batch_size: int = 1024,
):
    dev = torch.device(device)

    df = pd.read_csv(csv_path, usecols=["source", "destination", "timestamp"])
    df = df.sort_values("timestamp", kind="mergesort").reset_index(drop=True)

    # map node ids -> 0..n-1
    nodes = pd.Index(pd.concat([df["source"], df["destination"]], axis=0).unique())
    node2id = {node: i for i, node in enumerate(nodes.tolist())}
    num_nodes = len(node2id)

    # tensors
    src = torch.tensor([node2id[x] for x in df["source"].tolist()], device=dev, dtype=torch.long)
    dst = torch.tensor([node2id[x] for x in df["destination"].tolist()], device=dev, dtype=torch.long)
    ts  = torch.tensor(df["timestamp"].astype(float).to_numpy(), device=dev, dtype=dtype)

    # init W_0
    W = torch.randn((num_nodes, k_dim), device=dev, dtype=dtype)
    W = orthonormalize_qr(W)

    edge_weights = ExpDecayEdgeWeights(beta_e=beta_edge)

    Ws: List[torch.Tensor] = []
    batch_end_times: List[float] = []

    num_edges = src.numel()
    if batch_size <= 0:
        raise ValueError("batch_size must be positive.")


    for start in range(0, num_edges, batch_size):
        end = min(num_edges, start + batch_size)
        batch_id = start // batch_size
        print(f"[batch {batch_id}] edges {start}:{end}/{num_edges} "
            f"t_range=[{float(ts[start].item()):.3f}, {float(ts[end-1].item()):.3f}] "
            f"B={end-start}")
        src_t = src[start:end]
        dst_t = dst[start:end]
        ts_t  = ts[start:end]
        t_end = float(ts_t[-1].item())

        for u, v, tt in zip(src_t.tolist(), dst_t.tolist(), ts_t.tolist()):
            edge_weights.touch(u, v, float(tt), inc=1.0)

        W = oja_update_Wt(
            W=W,
            edge_weights=edge_weights,
            t=t_end,
            num_nodes=num_nodes,
            alpha=alpha,
            iters=iters_per_batch,
            device=dev,
            dtype=dtype,
            reorth_every=1,
        )

        batch_end_times.append(t_end)
        if store_all_W:
            Ws.append(W.detach().clone())

    return node2id, batch_end_times, (Ws if store_all_W else W)


In [8]:
import numpy as np
print((np.diff(links.timestamp)))

[0.00124113 0.04020174 0.00011623 ... 0.         0.         0.        ]


In [ ]:
import numpy as np 


beta = math.log(2)/np.average(np.diff(links.timestamp))
print(beta)

29053.264073170107


In [ ]:
if __name__ == "__main__":
    node2id, times, Ws, activity = run_temporal_oja_from_csv(
        #csv_path="syn_data/syn_net_p1.0_mu0.2_1.csv",
        csv_path="data/patent.csv",
        k_dim=16,
        alpha=0.05,
        beta_edge=beta,
        device="cpu",
        dtype=torch.float32,
        store_all_W=True,
        batch_size=1
    )
    print("num_nodes =", len(node2id))
    print("num_times =", len(times))
    print("W_t shape =", Ws[-1].shape)  # [n, k]

[batch 0] edges 0:1/41916 t_range=[0.000, 0.000] B=1
[batch 1] edges 1:2/41916 t_range=[0.001, 0.001] B=1
[batch 2] edges 2:3/41916 t_range=[0.041, 0.041] B=1
[batch 3] edges 3:4/41916 t_range=[0.042, 0.042] B=1
[batch 4] edges 4:5/41916 t_range=[0.042, 0.042] B=1
[batch 5] edges 5:6/41916 t_range=[0.042, 0.042] B=1
[batch 6] edges 6:7/41916 t_range=[0.042, 0.042] B=1
[batch 7] edges 7:8/41916 t_range=[0.042, 0.042] B=1
[batch 8] edges 8:9/41916 t_range=[0.042, 0.042] B=1
[batch 9] edges 9:10/41916 t_range=[0.042, 0.042] B=1
[batch 10] edges 10:11/41916 t_range=[0.042, 0.042] B=1
[batch 11] edges 11:12/41916 t_range=[0.043, 0.043] B=1
[batch 12] edges 12:13/41916 t_range=[0.043, 0.043] B=1
[batch 13] edges 13:14/41916 t_range=[0.043, 0.043] B=1
[batch 14] edges 14:15/41916 t_range=[0.043, 0.043] B=1
[batch 15] edges 15:16/41916 t_range=[0.044, 0.044] B=1
[batch 16] edges 16:17/41916 t_range=[0.044, 0.044] B=1
[batch 17] edges 17:18/41916 t_range=[0.044, 0.044] B=1
[batch 18] edges 18:1

: 

In [11]:
print(Ws[0].shape)

torch.Size([12214, 16])


In [ ]:
import pandas as pd
import torch

def first_seen_batch_from_csv(csv_path: str, node2id: dict, batch_size: int) -> torch.Tensor:
    df = pd.read_csv(csv_path, usecols=["source","destination","timestamp"])
    df = df.sort_values("timestamp", kind="mergesort").reset_index(drop=True)

    src = torch.tensor([node2id[x] for x in df["source"].tolist()], dtype=torch.long)
    dst = torch.tensor([node2id[x] for x in df["destination"].tolist()], dtype=torch.long)

    num_nodes = len(node2id)
    first = torch.full((num_nodes,), -1, dtype=torch.long)

    num_edges = src.numel()
    for e in range(num_edges):
        b = e // batch_size
        u = int(src[e].item())
        v = int(dst[e].item())
        if first[u] < 0: first[u] = b
        if first[v] < 0: first[v] = b

    return first

import torch

@torch.no_grad()
def aggregate_global_embedding(
    Ws,  # iterable of W_b, each [n,k] (可以是 list，也可以是生成器)
    first_seen_batch: torch.Tensor,  # [n] LongTensor
) -> torch.Tensor:
    # device/dtype from first W
    Ws_iter = iter(Ws)
    W0 = next(Ws_iter)  # 先拿一个确定形状
    n, k = W0.shape
    device = W0.device
    dtype = W0.dtype

    first_seen = first_seen_batch.to(device)
    sum_z = torch.zeros((n, k), device=device, dtype=dtype)
    cnt = torch.zeros((n,), device=device, dtype=dtype)

    # 把第 0 个也算进去
    b = 0
    valid = (first_seen >= 0) & (b >= first_seen)
    sum_z[valid] += W0[valid]
    cnt[valid] += 1

    # 继续流式处理后面的 W_b
    for b, W in enumerate(Ws_iter, start=1):
        print(b)
        valid = (first_seen >= 0) & (b >= first_seen)
        sum_z[valid] += W[valid]
        cnt[valid] += 1

    Z = sum_z / cnt.clamp_min(1.0).unsqueeze(1)
    return Z

In [18]:
first_seen = first_seen_batch_from_csv(
    csv_path="data/patent.csv",
    node2id=node2id,
    batch_size=1,
)

In [19]:
Z_global = aggregate_global_embedding(
    Ws=Ws,
    first_seen_batch=first_seen,
)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277


In [39]:
import numpy as np
from sklearn.cluster import KMeans

K = 6  # 你希望的社区数
X = Z_global.detach().cpu().numpy()

labels = KMeans(n_clusters=K, n_init=12, random_state=0).fit_predict(X)

In [34]:
import numpy as np
from sklearn.mixture import GaussianMixture

K = 6  # 你希望的社区数
X = Z_global.detach().cpu().numpy()

gmm = GaussianMixture(
    n_components=K,
    covariance_type="full",   # 也可以试 "diag"
    random_state=0
)

labels = gmm.fit_predict(X)
# labels: np.ndarray [num_nodes]，每个节点一个全局标签

In [40]:
import pandas as pd

def save_node_labels_csv(
    labels,
    out_csv_path: str = "global_labels.csv",
    *,
    id2node: dict | None = None,
):
    """
    Save (node, label) to a CSV.

    Args:
        labels: 1D array-like (list/np.ndarray/torch.Tensor) of length N.
                labels[i] is the label for node id i (0..N-1).
        out_csv_path: output csv path.
        id2node: optional mapping {int_id -> original_node}. If None, uses int ids.

    Output CSV columns:
        node,label
    """
    # convert labels to plain python list
    if hasattr(labels, "detach"):  # torch tensor
        labels_list = labels.detach().cpu().tolist()
    elif hasattr(labels, "tolist"):  # numpy
        labels_list = labels.tolist()
    else:
        labels_list = list(labels)

    N = len(labels_list)

    if id2node is None:
        nodes = list(range(N))
    else:
        # assume id2node covers 0..N-1
        nodes = [id2node[i] for i in range(N)]

    df = pd.DataFrame({"node": nodes, "label": labels_list})
    df.to_csv(out_csv_path, index=False)
    print(f"Saved: {out_csv_path} (rows={len(df)})")


id2node = {i: node for node, i in node2id.items()}
save_node_labels_csv(labels, "global_labels.csv", id2node=id2node)

Saved: global_labels.csv (rows=12214)


In [41]:
import pandas as pd
from sklearn.metrics import adjusted_mutual_info_score, normalized_mutual_info_score


df_g = pd.read_csv("global_labels.csv") 

df_t = pd.read_csv("data/node2label.txt", sep=r" ", header=None, names=["node", "label_txt"])


df_m = df_g.merge(df_t, on="node", how="inner")

ami = adjusted_mutual_info_score(df_m["label_txt"].to_numpy(), df_m["label"].to_numpy())
nmi = normalized_mutual_info_score(df_m["label_txt"].to_numpy(), df_m["label"].to_numpy())
print("matched_nodes =", len(df_m), "/", len(df_g), "(global) and", len(df_t), "(txt)")
print("AMI =", ami)
print("NMI =", nmi)

matched_nodes = 12214 / 12214 (global) and 12214 (txt)
AMI = 0.16978244450663801
NMI = 0.17044969643243163


In [31]:
import pandas as pd
from sklearn.metrics import adjusted_rand_score

df_g = pd.read_csv("global_labels.csv")

df_t = pd.read_csv(
    "data/node2label.txt",
    sep=r"\s+",
    header=None,
    names=["node", "label_txt"]
)

df_m = df_g.merge(df_t, on="node", how="inner")

ari = adjusted_rand_score(
    df_m["label_txt"].to_numpy(),
    df_m["label"].to_numpy()
)

print("matched_nodes =", len(df_m), "/", len(df_g), "(global) and", len(df_t), "(txt)")
print("ARI =", ari)

matched_nodes = 12214 / 12214 (global) and 12214 (txt)
ARI = 0.0997696081859564
